**Note**: The dataset is already splitted from the roboflow website. No need to fix it anymore!

In [1]:
import os
import random
import shutil
from tqdm import tqdm

In [2]:
def split_yolo_dataset_per_split(
    images_dir,
    labels_dir,
    output_dir,
    train_ratio=0.85,
    val_ratio=0.10,
    test_ratio=0.05,
    seed=42,
    img_exts=('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'),
    copy=True  # if False, files will be moved instead of copied
):
    """
    Split YOLO segmentation dataset into train/val/test where each split has its own images/ and labels/ subfolders.
    """

    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1.0"
    os.makedirs(output_dir, exist_ok=True)

    # collect image files
    all_images = [
        f for f in os.listdir(images_dir)
        if os.path.isfile(os.path.join(images_dir, f)) and f.lower().endswith(tuple(e.lower() for e in img_exts))
    ]
    all_images.sort()
    random.seed(seed)
    random.shuffle(all_images)

    n_total = len(all_images)
    n_train = int(n_total * train_ratio)
    n_val = int(n_total * val_ratio)
    n_test = n_total - n_train - n_val

    train_files = all_images[:n_train]
    val_files = all_images[n_train:n_train + n_val]
    test_files = all_images[n_train + n_val:]

    splits = {"train": train_files, "val": val_files, "test": test_files}

    # make folders
    for split in splits:
        os.makedirs(os.path.join(output_dir, split, "images"), exist_ok=True)
        os.makedirs(os.path.join(output_dir, split, "labels"), exist_ok=True)

    op_name = "copy" if copy else "move"

    # tqdm progress bar across all files
    total_files = sum(len(files) for files in splits.values())
    with tqdm(total=total_files, desc="Splitting dataset", unit="file") as pbar:
        for split, files in splits.items():
            for img_file in files:
                src_img = os.path.join(images_dir, img_file)
                dst_img = os.path.join(output_dir, split, "images", img_file)

                label_file = os.path.splitext(img_file)[0] + ".txt"
                src_lbl = os.path.join(labels_dir, label_file)
                dst_lbl = os.path.join(output_dir, split, "labels", label_file)

                try:
                    if copy:
                        shutil.copy2(src_img, dst_img)
                    else:
                        shutil.move(src_img, dst_img)
                except Exception as e:
                    print(f"Error while {op_name}ing image {src_img} -> {dst_img}: {e}")
                    pbar.update(1)
                    continue

                if os.path.exists(src_lbl):
                    try:
                        if copy:
                            shutil.copy2(src_lbl, dst_lbl)
                        else:
                            shutil.move(src_lbl, dst_lbl)
                    except Exception as e:
                        print(f"Error while {op_name}ing label {src_lbl} -> {dst_lbl}: {e}")
                else:
                    print(f"Warning: no label for {img_file} (expected {label_file})")

                pbar.update(1)

    print("\nSplit summary:")
    print(f"  total images = {n_total}")
    print(f"  train = {len(train_files)}")
    print(f"  val   = {len(val_files)}")
    print(f"  test  = {len(test_files)}")
    print(f"Saved to: {os.path.abspath(output_dir)}")

    return {
        "total": n_total,
        "train": len(train_files),
        "val": len(val_files),
        "test": len(test_files),
        "out_dir": os.path.abspath(output_dir)
    }

In [3]:
images_dir = "/home/c/choton/beemachine/everyday_test/oct1_25/partwhole_yolo_original_fixed/images"   # folder with all images
labels_dir = "/home/c/choton/beemachine/everyday_test/oct1_25/partwhole_yolo_original_fixed/labels"   # folder with all .txt labels
output_dir = "/home/c/choton/beemachine/everyday_test/oct1_25/partwhole_yolo_original_spitted"    # destination

stats = split_yolo_dataset_per_split(images_dir, labels_dir, output_dir, copy=True)

Splitting dataset: 100%|██████████| 4687/4687 [00:10<00:00, 431.42file/s]


Split summary:
  total images = 4687
  train = 3983
  val   = 468
  test  = 236
Saved to: /home/c/choton/beemachine/everyday_test/oct1_25/partwhole_yolo_original_spitted
